# Real FlashAttention (Colab GPU) vs CUDA-free FlashAttention (Apple M3 CPU)

This notebook runs **actual FlashAttention** on the Colab GPU via PyTorch's
`SDPBackend.FLASH_ATTENTION` and compares it, shape-for-shape, against the
AMX-based CPU kernel benchmarked on an Apple M3 (those numbers are baked in below).

Notes for an honest comparison:
- The CPU kernel cannot run here — it uses Apple's Accelerate/AMX, macOS-only.
- GPU FlashAttention only supports fp16/bf16; the M3 numbers are fp32 (the CPU
  kernel's native precision). The GPU gets its best case, the CPU its native one.
- PyTorch's flash backend needs compute capability >= 8.0 (Ampere+: A100, L4).
  On a free-tier **T4** (Turing, sm75) real FlashAttention cannot run at all —
  the notebook then uses the memory-efficient backend and labels it as such.

**Runtime > Change runtime type > GPU**, then Run all.

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU: Runtime > Change runtime type > GPU"
cap = torch.cuda.get_device_capability(0)
gpu = torch.cuda.get_device_name(0)
print(f"torch {torch.__version__} | CUDA {torch.version.cuda} | {gpu} (sm{cap[0]}{cap[1]})")
HAS_FLASH = cap >= (8, 0)
print("Real FlashAttention supported:", HAS_FLASH)

In [ ]:
import time
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

backend = SDPBackend.FLASH_ATTENTION if HAS_FLASH else SDPBackend.EFFICIENT_ATTENTION
label = "FlashAttention (CUDA)" if HAS_FLASH else "mem-efficient attention (T4 fallback; flash needs Ampere+)"
print("GPU backend:", label)

def bench_gpu(S, D=64, causal=False, B=1, H=8, dtype=torch.float16, reps=30):
    q, k, v = (torch.randn(B, H, S, D, device="cuda", dtype=dtype) for _ in range(3))
    with sdpa_kernel([backend]):
        for _ in range(5):
            F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(reps):
            F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / reps * 1e3  # ms

gpu_ms = {}
for causal in (False, True):
    for S in (256, 512, 1024, 2048, 4096, 8192):
        gpu_ms[(S, causal)] = bench_gpu(S, causal=causal)
        print(f"S={S:5d} causal={int(causal)}: {gpu_ms[(S, causal)]:8.3f} ms")

In [ ]:
# Apple M3 results (fp32, B=1, H=8, D=64), measured 2026-11-19:
# "ours" = AMX/Accelerate FlashAttention kernel; "torch_cpu" = PyTorch's
# flash-attention CPU kernel at its best config (8 threads).
m3_ours = {(256,False):0.51,(512,False):1.62,(1024,False):5.00,(2048,False):20.47,
           (4096,False):80.66,(8192,False):336.40,(256,True):0.50,(512,True):1.33,
           (1024,True):3.64,(2048,True):12.98,(4096,True):52.16,(8192,True):203.93}
m3_torch_cpu = {(256,False):0.79,(512,False):2.29,(1024,False):8.87,(2048,False):26.36,
                (4096,False):111.14,(8192,False):435.60,(256,True):0.87,(512,True):2.97,
                (1024,True):7.90,(2048,True):21.67,(4096,True):73.32,(8192,True):244.30}

hdr = f"{'S':>6} {'causal':>6} | {'GPU ms':>9} {'M3 ours ms':>11} {'M3 torch ms':>11} | {'GPU vs M3-ours':>14}"
print(gpu, "|", label)
print(hdr); print("-" * len(hdr))
for causal in (False, True):
    for S in (256, 512, 1024, 2048, 4096, 8192):
        g, o, t = gpu_ms[(S, causal)], m3_ours[(S, causal)], m3_torch_cpu[(S, causal)]
        print(f"{S:>6} {int(causal):>6} | {g:>9.3f} {o:>11.2f} {t:>11.2f} | {o/g:>13.1f}x")
print()
print("Reading the last column: how many times faster the Colab GPU is than the")
print("M3 CPU kernel. The M3 kernel's claim was never to beat a GPU — it was to")
print("beat the best attention available without CUDA, which it does (middle two")
print("columns).")